# PolicyProof AI — independent native AuditKIT CPU replay

This replays the **official AuditKIT 1.1.3** over previously saved base and 40-step trained-adapter responses from the original 48-case GPU evaluation ZIP. **It does not run Tiny Aya, download model weights, train a model, or verify human factual accuracy.** The examples are synthetic and already inspected.

Use a **CPU runtime** in Colab. The output contains base/tuned identities and scores, so **keep it private until both blinded reviewers have completed their independent reviews**. Do not share with your reviewers or upload the output to public GitHub.

## 1 — Clone latest project code; install the official evaluator

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

root = Path("/content/PolicyProof-AI")
if not root.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
        str(root),
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "auditkit==1.1.3"
], check=True)
subprocess.run([
    sys.executable, "-m", "scripts.prepare"
], cwd=root, check=True)
print("CPU environment prepared. No Tiny Aya model or GPU was loaded.")

## 2 — Upload the original paired-results ZIP

Upload `policyproof_dev48_paired_gpu_results.zip`, **not** the 40-step training ZIP. The verifier defaults to the exact SHA-256 from the user's original audited upload and refuses any mismatch. Do not change that SHA merely to make the command run.

In [ ]:
from google.colab import files
uploaded = files.upload()
names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(names) != 1:
    raise ValueError("Exactly one original paired-results ZIP must be uploaded")
archive = Path("/content") / names[0]
assert archive.is_file()
print("Received", archive.name, archive.stat().st_size, "bytes")

## 3 — Preflight: verify original ZIP, manifest, all 96 saved responses and their scores

This phase is read-only. It does not call AuditKIT, and it creates no private result.

In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.replay_dev48_native_auditkit",
    "--bundle", str(archive)
], cwd=root, check=True)
print("Original evidence checks passed. Official AuditKIT has NOT run yet.")

## 4 — Opt in to run native AuditKIT 1.1.3 (CPU only)

This replays two precomputed 48-response runs through the actual library. It verifies the fresh per-case results match the already archived counts and saves a **private owner-only** receipt in `reviews/`. It never overwrites previous evidence.

In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.replay_dev48_native_auditkit",
    "--bundle", str(archive), "--execute"
], cwd=root, check=True)
receipt = root / "reviews/dev48_native_auditkit_cpu_owner_only.json"
result = __import__("json").loads(receipt.read_text(encoding="utf-8"))
assert result["auditkit_executed_in_this_process"] is True
assert result["replayed_model_outputs_total"] == 96
assert result["semantic_correctness_human_review_completed"] is False
print("Native AuditKIT saved-output replay verified:", result["auditkit_version_replayed"])

## 5 — Download the private receipt; do not send it to blinded reviewers

This proves the local machine ran the official evaluator on the preserved saved text, **not** that a new GPU inference happened or that the trained model's answers were factually correct. Save alongside the unmodified source ZIP and use the independent reviewer process separately.

In [ ]:
files.download(str(receipt))
print("Do not publish owner-only base/tuned results until blinded review is complete.")